In [ ]:
"""
22_oneclass_svm.py
==================================================================

ONE-CLASS SVM

Dataset:
Wine Dataset

REAL WORLD WORKFLOW

1. Data Understanding
2. EDA
3. Feature Selection
4. Scaling
5. Baseline One-Class SVM
6. Anomaly Detection
7. Decision Function Analysis
8. Validation
9. Hyperparameter Tuning
10. PCA Visualization
11. Anomaly Profiling
12. Support Vector Analysis
13. Stability Analysis
14. Deployment Readiness
15. Interview Questions

==================================================================
"""

# ==========================================================
# STEP 0 : IMPORTS
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine

from sklearn.preprocessing import StandardScaler

from sklearn.svm import OneClassSVM

from sklearn.decomposition import PCA

import joblib


# ==========================================================
# STEP 1 : DATA UNDERSTANDING
# ==========================================================

wine = load_wine()

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

print("\nHEAD")
print(df.head())

print("\nINFO")
print(df.info())

print("\nDESCRIBE")
print(df.describe())

print("\nSHAPE")
print(df.shape)


# ==========================================================
# STEP 2 : EDA
# ==========================================================

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicates")
print(df.duplicated().sum())

plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(),
    cmap="coolwarm"
)

plt.title("Correlation Matrix")

plt.show()


# ==========================================================
# STEP 3 : FEATURE SELECTION
# ==========================================================

X = df.copy()


# ==========================================================
# STEP 4 : SCALING
# ==========================================================

"""
ABSOLUTELY MANDATORY

Unlike Isolation Forest,
One-Class SVM is distance based.

Never skip scaling.
"""

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ==========================================================
# STEP 5 : BASELINE MODEL
# ==========================================================

ocsvm = OneClassSVM(

    kernel='rbf',

    gamma='scale',

    nu=0.05

)

"""
IMPORTANT PARAMETERS

kernel
------
rbf (most common)

gamma
------
Kernel coefficient

scale (recommended)

nu
--
Expected anomaly proportion

0 < nu <= 1

Acts similar to contamination
in Isolation Forest
"""

# ==========================================================
# STEP 6 : ANOMALY DETECTION
# ==========================================================

labels = ocsvm.fit_predict(
    X_scaled
)

"""
OUTPUT

1  -> Normal

-1 -> Anomaly
"""

df["anomaly"] = labels

print("\nAnomaly Counts")

print(
    df["anomaly"].value_counts()
)


# ==========================================================
# STEP 7 : DECISION FUNCTION ANALYSIS
# ==========================================================

scores = ocsvm.decision_function(
    X_scaled
)

df["anomaly_score"] = scores

print("\nLowest Scores")

print(

    df.sort_values(
        "anomaly_score"
    )

    .head(10)

)

"""
Negative Score

=
Outside Boundary

Potential Anomaly

Positive Score

=
Inside Boundary

Normal Observation
"""


# ==========================================================
# STEP 8 : VALIDATION
# ==========================================================

anomaly_count = np.sum(
    labels == -1
)

anomaly_percent = (

    anomaly_count
    /
    len(df)

) * 100

print("\nAnomaly Count")

print(anomaly_count)

print("\nAnomaly Percent")

print(
    round(
        anomaly_percent,
        2
    ),
    "%"
)

# ==========================================================
# STEP 9 : HYPERPARAMETER TUNING
# ==========================================================

results = []

for nu in [

    0.01,
    0.03,
    0.05,
    0.10,
    0.15

]:

    model = OneClassSVM(

        kernel='rbf',

        gamma='scale',

        nu=nu

    )

    lbl = model.fit_predict(
        X_scaled
    )

    anomalies = np.sum(
        lbl == -1
    )

    results.append([

        nu,

        anomalies

    ])

tuning_df = pd.DataFrame(

    results,

    columns=[

        "nu",

        "anomaly_count"

    ]

)

print("\nNU Analysis")

print(tuning_df)


# ==========================================================
# STEP 10 : PCA VISUALIZATION
# ==========================================================

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_scaled
)

plt.figure(figsize=(10,6))

plt.scatter(

    X_pca[df["anomaly"]==1,0],

    X_pca[df["anomaly"]==1,1],

    label="Normal"

)

plt.scatter(

    X_pca[df["anomaly"]==-1,0],

    X_pca[df["anomaly"]==-1,1],

    label="Anomaly"

)

plt.xlabel("PC1")

plt.ylabel("PC2")

plt.title("One-Class SVM Results")

plt.legend()

plt.show()


# ==========================================================
# STEP 11 : ANOMALY PROFILING
# ==========================================================

normal_data = df[
    df["anomaly"] == 1
]

anomaly_data = df[
    df["anomaly"] == -1
]

print("\nNormal Mean")

print(

    normal_data.mean(
        numeric_only=True
    )

)

print("\nAnomaly Mean")

print(

    anomaly_data.mean(
        numeric_only=True
    )

)

"""
Industry Goal:

Understand WHY
these records are anomalous.
"""


# ==========================================================
# STEP 12 : SUPPORT VECTOR ANALYSIS
# ==========================================================

"""
Unique to SVM

Support vectors define
the anomaly boundary.
"""

print("\nNumber of Support Vectors")

print(
    len(
        ocsvm.support_
    )
)

print("\nSupport Vector Ratio")

print(

    len(ocsvm.support_)
    /
    len(X_scaled)

)


# ==========================================================
# STEP 13 : STABILITY ANALYSIS
# ==========================================================

gamma_values = [

    'scale',

    0.01,

    0.1,

    1

]

for gamma in gamma_values:

    model = OneClassSVM(

        kernel='rbf',

        gamma=gamma,

        nu=0.05

    )

    lbl = model.fit_predict(
        X_scaled
    )

    anomalies = np.sum(
        lbl == -1
    )

    print(

        f"gamma={gamma}"

        f" -> anomalies={anomalies}"

    )


# ==========================================================
# STEP 14 : DEPLOYMENT READINESS
# ==========================================================

joblib.dump(

    ocsvm,

    "oneclass_svm.pkl"

)

joblib.dump(

    scaler,

    "oneclass_svm_scaler.pkl"

)

print("\nArtifacts Saved")

print("oneclass_svm.pkl")

print("oneclass_svm_scaler.pkl")


# Example Prediction

sample = X.iloc[[0]]

sample_scaled = scaler.transform(
    sample
)

prediction = ocsvm.predict(
    sample_scaled
)

score = ocsvm.decision_function(
    sample_scaled
)

print("\nPrediction")

print(prediction)

print("\nScore")

print(score)


# ==========================================================
# STEP 15 : INTERVIEW QUESTIONS
# ==========================================================

"""
1. What is One-Class SVM?

2. Why is it called One-Class?

3. How does One-Class SVM work?

4. Difference between SVM and One-Class SVM?

5. What is anomaly detection?

6. What is kernel trick?

7. Why RBF kernel is popular?

8. What is gamma?

9. What is nu?

10. What happens if gamma is too high?

11. What happens if gamma is too low?

12. What happens if nu increases?

13. What are support vectors?

14. Why scaling is mandatory?

15. One-Class SVM vs Isolation Forest?

16. One-Class SVM vs DBSCAN?

17. What is decision_function()?

18. Why negative score indicates anomaly?

19. Can One-Class SVM handle nonlinear boundaries?

20. Why One-Class SVM struggles on large datasets?

21. Time Complexity?

22. Real-world use cases?

23. Fraud Detection use case?

24. Cyber Security use case?

25. Explain One-Class SVM end-to-end.
"""

# ==========================================================
# INTERVIEW CHEAT SHEET
# ==========================================================

"""
ISOLATION FOREST

Idea:
-----
Isolate anomalies using trees

Advantages:
-----------
Fast
Scalable
High-dimensional data

--------------------------------------------------

ONE-CLASS SVM

Idea:
-----
Learn boundary around normal data

Advantages:
-----------
Complex nonlinear boundaries

Disadvantages:
--------------
Slow
Memory intensive
Needs scaling

--------------------------------------------------

Common Interview Question:

Which one would you use
for 10 million records?

Answer:

Isolation Forest

because One-Class SVM
does not scale well.
"""

# ==========================================================
# END OF FILE
# ==========================================================